In [0]:
from pyspark.sql.functions import col, explode, from_json, expr
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, DoubleType, MapType

# 1. Definicja schematu dla mocno zagnieżdżonego JSON-a z Overpass API
schema = StructType([
    StructField("elements", ArrayType(
        StructType([
            StructField("lat", DoubleType()),
            StructField("lon", DoubleType()),
            StructField("tags", MapType(StringType(), StringType()))
        ])
    ))
])

# Ścieżka do naszego zrzutu (warstwa Bronze)
storage_account_name = "adlsmaritimegen2"
bronze_ports_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/osm_ports_raw/ports_europe.json"

# 2. Wczytanie pliku tekstowego zawierającego JSON
df_raw = spark.read.text(bronze_ports_path)

# 3. Parsowanie JSON-a i eksplozja tablicy do płaskich wierszy
df_parsed = df_raw.select(from_json(col("value"), schema).alias("data"))
df_exploded = df_parsed.select(explode(col("data.elements")).alias("element"))

# 4. Wyciągnięcie konkretnych wartości z węzłów i tagów OSM
df_silver_ports = df_exploded.select(
    expr("element.tags['name']").alias("port_name"),
    col("element.lat").alias("latitude"),
    col("element.lon").alias("longitude")
)

# Filtrujemy rekordy bez współrzędnych i bez nazwy portu
df_silver_ports = df_silver_ports.filter(col("latitude").isNotNull() & col("longitude").isNotNull() & col("port_name").isNotNull())

# 5. Inżynieria Przestrzenna (Uber H3)
# Dodajemy indeks H3 na podstawie wyciągniętych współrzędnych (i ZACHOWUJEMY JE w wyniku)
df_silver_enriched = df_silver_ports.selectExpr(
    "port_name",
    "'EU' as country_code", 
    "latitude",
    "longitude",
    "h3_longlatash3(longitude, latitude, 7) AS port_h3_id_res7"
)

# 6. Zapis z opcją wymuszenia nadpisania schematu (overwriteSchema = true)
df_silver_enriched.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbw_maritime_2026.maritime_showcase.dim_world_ports")

print("Zbudowano nową tabelę!")